# Train Word2Vec on Your Own Text

This notebook trains a Word2Vec model from scratch on a text file you provide.

**How to use:**
1. Place your text file (`.txt`) anywhere accessible
2. Set the `TEXT_FILE` path in cell 2
3. Run all cells

---

In [1]:
%pip install -q nltk gensim matplotlib scikit-learn numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
import nltk
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)

import re
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from gensim.models import Word2Vec
from sklearn.decomposition import PCA

## 1 — Configuration

Point `TEXT_FILE` to your `.txt` file. Adjust hyperparameters below.

In [3]:
# ── Path to your text file ──────────────────────────────────
TEXT_FILE = "corpus.txt"  # <-- change this

# ── Preprocessing ───────────────────────────────────────────
LANGUAGE = "english"        # stopword language
REMOVE_STOPWORDS = False    # set True to strip stop words before training
MIN_WORD_LENGTH = 2         # drop very short tokens

# ── Word2Vec hyperparameters ────────────────────────────────
VECTOR_SIZE = 100   # embedding dimensions
WINDOW = 5          # context window (words on each side)
MIN_COUNT = 3       # ignore words that appear fewer than this
SG = 1              # 1 = skip-gram, 0 = CBOW
EPOCHS = 20         # training passes over the corpus
WORKERS = 4         # parallel threads

## 2 — Load & Inspect the Text

In [4]:
path = Path(TEXT_FILE)
assert path.exists(), f"File not found: {path.resolve()}"

raw_text = path.read_text(encoding="utf-8")

print(f"File       : {path.name}")
print(f"Size       : {len(raw_text):,} characters")
print(f"Lines      : {raw_text.count(chr(10)):,}")
print(f"\n--- First 500 characters ---")
print(raw_text[:500])

AssertionError: File not found: /home/khairi/work/personal/workshops/nlp/intro-nlp/notebooks/corpus.txt

## 3 — Tokenize into Sentences & Words

In [ ]:
stop_words = set(stopwords.words(LANGUAGE)) if REMOVE_STOPWORDS else set()

def preprocess(text: str) -> list[list[str]]:
    """Split text into sentences of clean tokens."""
    sentences = sent_tokenize(text)
    processed = []
    for sent in sentences:
        tokens = word_tokenize(sent.lower())
        tokens = [
            t for t in tokens
            if t.isalpha()
            and len(t) >= MIN_WORD_LENGTH
            and t not in stop_words
        ]
        if tokens:
            processed.append(tokens)
    return processed

corpus = preprocess(raw_text)

print(f"Sentences  : {len(corpus):,}")
print(f"Total tokens: {sum(len(s) for s in corpus):,}")
print(f"\nSample sentences:")
for s in corpus[:5]:
    print(f"  {s[:12]}{'...' if len(s) > 12 else ''}")

In [ ]:
# Word frequency distribution
from collections import Counter

all_tokens = [t for s in corpus for t in s]
freq = Counter(all_tokens)

print(f"Unique words: {len(freq):,}")
print(f"Words appearing >= {MIN_COUNT} times: {sum(1 for c in freq.values() if c >= MIN_COUNT):,}\n")
print("Top 25 most frequent words:")
for word, count in freq.most_common(25):
    bar = "█" * min(count // max(freq.most_common(25)[-1][1], 1), 40)
    print(f"  {word:15s} {count:6,d}  {bar}")

## 4 — Train Word2Vec

In [ ]:
%%time

model = Word2Vec(
    sentences=corpus,
    vector_size=VECTOR_SIZE,
    window=WINDOW,
    min_count=MIN_COUNT,
    sg=SG,
    epochs=EPOCHS,
    workers=WORKERS,
)

wv = model.wv
print(f"\nVocabulary : {len(wv):,} words")
print(f"Dimensions : {wv.vector_size}")
print(f"Algorithm  : {'Skip-gram' if SG else 'CBOW'}")

## 5 — Explore Your Embeddings

In [ ]:
# Pick a word that exists in your corpus
# --- Change this! ---
query = "the"  # replace with a word from your text

if query in wv:
    print(f"Top 10 neighbors of '{query}':\n")
    for i, (word, score) in enumerate(wv.most_similar(query, topn=10), 1):
        bar = "█" * int(score * 20) + "░" * (20 - int(score * 20))
        print(f"  {i:2d}. {word:20s} {bar} {score:.3f}")
else:
    print(f"'{query}' not in vocabulary. Try one of these:")
    print(list(wv.key_to_index.keys())[:20])

In [ ]:
# Similarity between two words from your corpus
word_a = "the"   # <-- change
word_b = "and"   # <-- change

if word_a in wv and word_b in wv:
    print(f"similarity('{word_a}', '{word_b}') = {wv.similarity(word_a, word_b):.3f}")
else:
    missing = [w for w in [word_a, word_b] if w not in wv]
    print(f"Not in vocabulary: {missing}")

## 6 — Visualize Embeddings (2D)

In [ ]:
def plot_words(word_list, wv, title="Word Vectors in 2D", figsize=(10, 7)):
    """Project word vectors to 2D with PCA and plot them."""
    present = [w for w in word_list if w in wv]
    missing = [w for w in word_list if w not in wv]
    if missing:
        print(f"Skipping (not in vocab): {missing}")
    if len(present) < 2:
        print("Need at least 2 words in vocabulary to plot.")
        return

    vectors = np.array([wv[w] for w in present])
    coords = PCA(n_components=2).fit_transform(vectors)

    fig, ax = plt.subplots(figsize=figsize)
    ax.scatter(coords[:, 0], coords[:, 1], s=60, alpha=0.7)
    for i, word in enumerate(present):
        ax.annotate(word, (coords[i, 0], coords[i, 1]),
                    fontsize=11, fontweight="bold",
                    xytext=(5, 5), textcoords="offset points")
    ax.set_title(title, fontsize=14)
    ax.grid(True, alpha=0.3)
    ax.set_xlabel("PC 1")
    ax.set_ylabel("PC 2")
    plt.tight_layout()
    plt.show()

In [ ]:
# Plot the 30 most frequent words
top_words = [word for word, _ in freq.most_common(50) if word in wv][:30]
plot_words(top_words, wv, title="Top 30 Words from Your Corpus")

In [ ]:
# Plot a custom group of words from your corpus
# --- Change these to words that appear in your text! ---
my_words = ["he", "she", "they", "said", "went", "could", "would"]
plot_words(my_words, wv, title="Custom Word Group")

## 7 — Compare: Skip-gram vs CBOW

Train both variants and see how their nearest neighbors differ.

In [ ]:
model_sg = Word2Vec(corpus, vector_size=VECTOR_SIZE, window=WINDOW,
                    min_count=MIN_COUNT, sg=1, epochs=EPOCHS, workers=WORKERS)
model_cbow = Word2Vec(corpus, vector_size=VECTOR_SIZE, window=WINDOW,
                      min_count=MIN_COUNT, sg=0, epochs=EPOCHS, workers=WORKERS)

# Pick a comparison word
compare_word = "the"  # <-- change to a word from your text

if compare_word in model_sg.wv and compare_word in model_cbow.wv:
    sg_neighbors = [w for w, _ in model_sg.wv.most_similar(compare_word, topn=10)]
    cbow_neighbors = [w for w, _ in model_cbow.wv.most_similar(compare_word, topn=10)]

    print(f"Neighbors of '{compare_word}':\n")
    print(f"  {'#':>2s}  {'Skip-gram':20s}  {'CBOW':20s}")
    print(f"  {'─'*2}  {'─'*20}  {'─'*20}")
    for i in range(10):
        sg_w = sg_neighbors[i] if i < len(sg_neighbors) else ""
        cbow_w = cbow_neighbors[i] if i < len(cbow_neighbors) else ""
        match = " ✓" if sg_w == cbow_w else ""
        print(f"  {i+1:2d}  {sg_w:20s}  {cbow_w:20s}{match}")
else:
    print(f"'{compare_word}' not in vocabulary.")

## 8 — Effect of Window Size

See how changing the context window affects what the model learns. Smaller windows capture **syntactic** similarity (same part of speech), larger windows capture **topical** similarity.

In [ ]:
compare_word = "the"  # <-- change to a word from your text
windows = [2, 5, 10]

for w_size in windows:
    m = Word2Vec(corpus, vector_size=VECTOR_SIZE, window=w_size,
                 min_count=MIN_COUNT, sg=SG, epochs=EPOCHS, workers=WORKERS)
    if compare_word in m.wv:
        neighbors = [word for word, _ in m.wv.most_similar(compare_word, topn=5)]
        print(f"  window={w_size:2d}  →  {neighbors}")
    else:
        print(f"  window={w_size:2d}  →  '{compare_word}' not in vocab")

## 9 — Save & Load Your Model

Save the trained model so you can reload it later without retraining.

In [ ]:
SAVE_PATH = "my_word2vec.model"

# Save full model (can resume training later)
model.save(SAVE_PATH)
print(f"Model saved to {SAVE_PATH}")

# Load it back
loaded_model = Word2Vec.load(SAVE_PATH)
print(f"Loaded model: {len(loaded_model.wv):,} words, {loaded_model.wv.vector_size}d")

In [ ]:
# Save vectors only (lightweight, read-only)
VECTORS_PATH = "my_word2vec.vectors.kv"
model.wv.save(VECTORS_PATH)
print(f"Vectors saved to {VECTORS_PATH}")

# Load vectors only
from gensim.models import KeyedVectors
loaded_wv = KeyedVectors.load(VECTORS_PATH)
print(f"Loaded vectors: {len(loaded_wv):,} words, {loaded_wv.vector_size}d")